# Cross-Pipeline Validation (Harmonized)

Computes all metrics across pipeline pairs:
- ARI, NMI (global clustering)
- Dice (per-cluster overlap)
- Jaccard + Spearman ρ (native DE + standardized DE)
- Module score profile correlation (biology preservation)

**Run AFTER** all pipeline notebooks have completed.

Uses Hungarian matching for optimal cluster correspondence.
Standardized DE isolates cluster-label drift from DE-engine drift.

In [1]:
# === CHANGE THIS TO SWITCH DATASETS ===
DATASET = "lung65k"   # or "lung65k"

CONFIG_PATH = "benchmark_config.json"

In [2]:
# The validation script is complex enough to run as-is.
# Execute it via subprocess so it uses argparse correctly.
import subprocess
result = subprocess.run(
    ["python", "validate_cross_pipeline_harmonized.py",
     "--dataset", DATASET, "--config", CONFIG_PATH],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Validation — Human Lung Cell 65K
Pipelines detected:
  scanpy_cpu   write/lung_65k_scanpy_cpu_harmonized_clusters.csv
  rsc_gpu_0141 write/lung_65k_rsc_gpu_0141_harmonized_clusters.csv
  seurat_cpu   write/lung_65k_seurat_cpu_harmonized_clusters.csv
  rsc_gpu_015  write/lung_65k_rsc_gpu_015_harmonized_clusters.csv

Summary:
            dataset                  comparison  common_cells  clusters_a  clusters_b      ARI      NMI  mean_dice  mean_native_deg_jaccard  mean_native_deg_spearman  mean_standardized_deg_jaccard  mean_standardized_deg_spearman  mean_module_profile_rho
Human Lung Cell 65K  scanpy_cpu vs rsc_gpu_0141         65462          38          37 0.873273 0.930866   0.844177                 0.834653                  0.914099                       0.834653                        0.914099                 0.967568
Human Lung Cell 65K    scanpy_cpu vs seurat_cpu         65462          38          44 0.703593 0.864938   0.696699                 0.454156                  0.857105 

In [3]:
# Load and display the summary
import json
import pandas as pd

with open(CONFIG_PATH) as f:
    dcfg = json.load(f)["datasets"][DATASET]
prefix = dcfg["pipeline_prefix"]
val_dir = f"write/validation_{prefix}_harmonized"

summary_csv = f"{val_dir}/{prefix}_validation_summary.csv"
try:
    df = pd.read_csv(summary_csv)
    print(df.to_string(index=False))
except FileNotFoundError:
    print(f"Summary not found: {summary_csv}")

            dataset                  comparison  common_cells  clusters_a  clusters_b      ARI      NMI  mean_dice  mean_native_deg_jaccard  mean_native_deg_spearman  mean_standardized_deg_jaccard  mean_standardized_deg_spearman  mean_module_profile_rho
Human Lung Cell 65K  scanpy_cpu vs rsc_gpu_0141         65462          38          37 0.873273 0.930866   0.844177                 0.834653                  0.914099                       0.834653                        0.914099                 0.967568
Human Lung Cell 65K    scanpy_cpu vs seurat_cpu         65462          38          44 0.703593 0.864938   0.696699                 0.454156                  0.857105                       0.700396                        0.843269                 0.915789
Human Lung Cell 65K   scanpy_cpu vs rsc_gpu_015         65462          38          38 0.902163 0.942271   0.908990                 0.890061                  0.966423                       0.890061                        0.966423          

In [4]:
# Load detailed results
detail_json = f"{val_dir}/{prefix}_validation_details.json"
try:
    with open(detail_json) as f:
        details = json.load(f)
    for pair, metrics in details.items():
        print(f"\n=== {pair} ===")
        print(f"  ARI: {metrics['ARI']:.4f}")
        print(f"  NMI: {metrics['NMI']:.4f}")
        print(f"  Mean Dice: {metrics.get('mean_dice', 'N/A')}")
        print(f"  Mean Std DEG Jaccard: {metrics.get('mean_standardized_deg_jaccard', 'N/A')}")
        print(f"  Mean Std DEG Spearman: {metrics.get('mean_standardized_deg_spearman', 'N/A')}")
        print(f"  Mean Module ρ: {metrics.get('mean_module_profile_rho', 'N/A')}")
except FileNotFoundError:
    print(f"Details not found: {detail_json}")


=== scanpy_cpu__vs__rsc_gpu_0141 ===
  ARI: 0.8733
  NMI: 0.9309
  Mean Dice: 0.8441765863400619
  Mean Std DEG Jaccard: 0.8346527412799657
  Mean Std DEG Spearman: 0.914099046696003
  Mean Module ρ: 0.9675675675675675

=== scanpy_cpu__vs__seurat_cpu ===
  ARI: 0.7036
  NMI: 0.8649
  Mean Dice: 0.6966991161419402
  Mean Std DEG Jaccard: 0.7003956044881893
  Mean Std DEG Spearman: 0.8432694269647765
  Mean Module ρ: 0.9157894736842105

=== scanpy_cpu__vs__rsc_gpu_015 ===
  ARI: 0.9022
  NMI: 0.9423
  Mean Dice: 0.9089900187278337
  Mean Std DEG Jaccard: 0.8900608879912886
  Mean Std DEG Spearman: 0.9664233828236726
  Mean Module ρ: 1.0

=== rsc_gpu_0141__vs__seurat_cpu ===
  ARI: 0.7265
  NMI: 0.8699
  Mean Dice: 0.7837601296614611
  Mean Std DEG Jaccard: 0.7809096217960632
  Mean Std DEG Spearman: 0.8933877825751376
  Mean Module ρ: 0.9513513513513514

=== rsc_gpu_0141__vs__rsc_gpu_015 ===
  ARI: 0.9273
  NMI: 0.9625
  Mean Dice: 0.8824679962499626
  Mean Std DEG Jaccard: 0.8797376693